# ಪಾಠ 05 - ಏಜೆಂಟಿಕ್ RAG


## ಸೆಟ್‌ಅಪ್

ಈ ನೋಟುಬುಕ್ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ ಬಳಸಿ ಏಜೆಂಟಿಕ್ RAG (Retrieval-Augmented Generation) ಪ್ಯಾಟರ್ನ್ ಅನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ.

**ಮೊದಲು ಬೇಕಾಗುವ ವಸ್ತುಗಳು:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ನಿಮ್ಮ ಆಜೂರ್ AI ಶೋಧನೆ ಸೇವೆ ಎಂಡ್‌ಪಾಯಿನ್‌ಟ್
- `AZURE_SEARCH_API_KEY` — ನಿಮ್ಮ ಆಜೂರ್ AI ಶೋಧನೆ API ಕೀ
- ಪರಿಸರ ಚರಗಳ ಮೂಲಕ ಸಂರಚಿಸಿದ ಆಜೂರ್ OpenAI ನಿಯೋಜನೆ
- ಆಜೂರ್ CLI ಪ್ರಾಮಾಣೀಕರಿಸಲಾಗಿದೆ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ಏಜೆಂಟಿಕ್ RAG ಎಂದರೆ ಏನು?

ಸಾಂಪ್ರದಾಯಿಕ RAG ಸ್ಥಿರ पाइಪ್‌ಲೈನ್ ಅನುಸರಿಸುತ್ತದೆ: ದಾಖಲಗಳನ್ನು ಪಡೆದು ನಂತರ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಉತ್ಪಾದಿಸುತ್ತದೆ. **ಏಜೆಂಟಿಕ್ RAG** ಏಜೆಂಟ್‌ಗೆ ಸ್ವಾಯತ್ತತೆಯನ್ನು ನೀಡುತ್ತದೆ, **ಎಲ್ಲಿ** ಮತ್ತು **ಯಾಕಿಂತ** ಮಾಹಿತಿ ಪಡೆದುಕೊಳ್ಳಬೇಕೆಂದು ಆಯ್ಕೆಮಾಡಲು.

ಏಜೆಂಟಿಕ್ RAG ಸಹಿತ, ಏಜೆಂಟ್ ಚನ್ನಾಗಿ ಮಾಡಬಹುದು:
- ಪ್ರಶ್ನೆಗೆ ಉತ್ತರಿಸುವ ಮೊದಲು ಮಾಹಿತಿ ಪಡೆದುಕೊಳ್ಳುವುದು ಅಗತ್ಯವೋ ಇಲ್ಲವೋ **ನಿರ್ಧರಿಸಲು**
- ಯಾವ ಡೇಟಾ ಮೂಲ ಅಥವಾ ಉಪಕರಣವನ್ನು ಕೇಳಬೇಕೆಂದು **ಆಯ್ಕೆಮಾಡಲು**
- ಪಡೆದ ಫಲಿತಾಂಶಗಳನ್ನು **ಮೌಲ್ಯಮಾಪನ ಮಾಡುವುದು** ಮತ್ತು ಮೊದಲ ಪ್ರಯತ್ನ ಸಂಪೂರ್ಣವಾಗಿರದಿದ್ದರೆ ತದನಂತರ ಮಾಹಿತಿ ಪಡೆದುಕೊಳ್ಳಲು
- ವಿವಿಧ ಮಾಹಿತಿ ಪಡೆಯುವ ಹಂತಗಳಿಂದ ಮಾಹಿತಿ **ಸರಣಿಬದ್ಧವಾಗಿ** ಸಂಯೋಜಿಸಲು

ಇದು ಏಜೆಂಟ್ ಅನ್ನು ಸ್ಥಿರ "ಪಡೆಯಿರಿ-ನಂತರ-ಉತ್ಪಾದಿಸಿ" पाइಪ್‌ಲೈನ್‌ಗಿಂತ ಹೆಚ್ಚಿನ ನಿಗದಿತತೆ ಮತ್ತು ಸರಳತೆ ನೀಡುತ್ತದೆ.


## ಶೋಧನೆ ಸಾಧನ ಸೃಷ್ಟಿಸುವುದು

Agentic RAG ನಲ್ಲಿ, బయటి ಡೇಟಾ ಮೂಲಗಳನ್ನು ಏಜೆಂಟ್ ಬೇಡಿಕೆಯಂತೆ ಕರೆಹೊಯ್ಯಬಹುದಾದ **ಸಾಧನಗಳಾಗಿ** ಮರುಹೊಂದಿಸಲಾಗಿದೆ. ಇದರಿಂದ ಏಜೆಂಟ್‌ಗೆ ಸಂಗ್ರಹಣೆಯನ್ನು ಕಡ್ಡಾಯ ಹಂತವಲ್ಲದೆ, ಬೇರೊಂದು ಕ್ರಮವೆಂದೇ ಪರಿಗಣಿಸಲು ಸಾಧ್ಯವಾಗುತ್ತದೆ.

ಕೆಳಗಿದೆ ನಾವು ಒಂದು ಪ್ರಯಾಣ ಜ್ಞಾನಾಧಾರವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ, ಅದನ್ನು ಏಜೆಂಟ್ ಗುರಿ ಸ್ಥಳದ ಮಾಹಿತಿಯನ್ನು ಹುಡುಕಲು ಕರೆಸಬಹುದಾದ ಸಾಧನವಾಗಿ ප්‍රಕಟಿಸುವುದು.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ಏಜೆಂಟ್ ನಿರ್ಮಿಸುವಿಕೆ

ಈಗ ನಾವು **ಪ್ರತಿನಿಧಾನ ನೀಡುವ ಮೊದಲೇ ಯಾವಾಗಲೂ ಮಾಹಿತಿ ಸಂಗ್ರಹಿಸುವಂತೆ** ನಿರ್ದೇಶಿಸಲಾಗಿದೆ ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುವೆವು. ಏಜೆಂಟ್ ತನ್ನ ಉತ್ತರಗಳನ್ನು ತನ್ನ ಸ್ವಂತ ತರಬೇತಿ ಡೇಟಾ ಆಧಾರಿಸಿಕೊಳ್ಳದೆ, Gsearch_travel_knowledge ಉನ್ನತದಿಂದ ಉತ್ತರಗಳನ್ನು ಮೂಲಗೊಳಿಸಲು ಉಪಯೋಗಿಸುತ್ತದೆ.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ಸಹಪಠನ ಮರುಪಡೆತ — ನಿರ್ಮಾಪಕ-ಪರಿಶೀಲಕ ಮಾದರಿ

Agentic RAG ರ ಪ್ರಮುಖ ಲಾಭವು **ಸಹಪಠನ ಮರುಪಡೆತ** ಆಗಿದೆ. ಏಜೆಂಟ್ ಆರಂಭಿಕ ಕಂಡುಹಿಡಿದುವುದನ್ನು ಪರಿಶೀಲಿಸಿ, ಪರಿಷ್ಕರಿಸಿ ಅಥವಾ ವಿಸ್ತರಿಸಲು ಹಲವಾರು ಬಾರಿ ಹುಡುಕಾಟ ನಡೆಸಬಹುದು — "ನಿರ್ಮಾಪಕ-ಪರಿಶೀಲಕ" ಕೆಲಸದ ನಿರ್ವಹಣೆಯಂತೆ:

1. **ನಿರ್ಮಾಪಕ ಹಂತ**: ಏಜೆಂಟ್ ಆರಂಭಿಕ ಮಾಹಿತಿಯನ್ನು ಮರುವೀರಿಸಿಕೊಂಡು ಉತ್ತರದ ಮೊದಲ ಮಸೂದೆ ರೂಪಿಸುತ್ತದೆ.
2. **ಪರಿಶೀಲಕ ಹಂತ**: ಏಜೆಂಟ್ ವಿವರಗಳನ್ನು ಪರಿಶೀಲಿಸಲು ಅಥವಾ ಖಾಲಿ ಸ್ಥಳಗಳನ್ನು ತುಂಬಿಸಲು ಹೆಚ್ಚುವರಿ ಮರುಪಡೆತ ಮಾಡುತ್ತದೆ.

ಕೆಳಗೆ, ಏಜೆಂಟ್ ಹಲವಾರು ಗಮ್ಯಸ್ಥಳಗಳನ್ನು ಹೋಲಿಸುವ ಪ್ರಶ್ನೆಯನ್ನು ಕೇಳಲಾಗಿದ್ದು, ಇದರಿಂದ ಅದು ಅನೇಕ ಸಮಯಗಳು ಹುಡುಕಲು ಪ್ರೇರಿತವಾಗುತ್ತದೆ.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಬಳಸಿ **ಏಜೆಂಟಿಕ್ RAG** ಸಿಸ್ಟಂ ಅನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸುವುದನ್ನು ಕಲಿತಿರಿ:

- **ಏಜೆಂಟಿಕ್ RAG** ಏಜೆಂಟ್‌ಗಳಿಗೆ ಮಾಹಿತಿ ಪಡೆಯುವುದನ್ನು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ನಿರ್ಧರಿಸಲು ಅನುಮತಿಸುತ್ತದೆ, ಇದರಿಂದ ಪುರಸ್ಕಾರ ಸ್ಥಿರವಾಗಿಲ್ಲದೆ ಚುರುಕಾಗಿ ನಡೆಯುತ್ತದೆ.
- **ಆಯ್ಕೆಗಳಲ್ಲಿ ಡೇಟಾ ಮೂಲಗಳು**: ಔಟ್‌ಸೈಡ್ ಜ್ಞಾನ ಆಧಾರಗಳು (ಆಜರ್ AI ಸರ್ಚ್ ಮುಂತಾದವು) ಟೂಲ್‌ಗಳಾಗಿ ಸುತ್ತುವ അതನ್ನು ಏಜೆಂಟ್ ಕರೆಯಬಹುದು.
- **ಪುನರಾವರ್ತಿತ ಪುರಸ್ಕಾರ**: ಮೇಕರ್-ಚೆಕರ ಮಾದರಿ ಏಜೆಂಟ್‌ಗೆ ಅನೇಕ ಗೊತ್ತುಮಾಡುವ ಹಂತಗಳನ್ನು ನಿರ್ವಹಿಸಲು ಅವಕಾಶ ನೀಡುತ್ತದೆ — ಹುಡುಕಾಟ, ಪರಿಶೀಲನೆ, ಮತ್ತು ಸಂಶೋಧನೆ ಮೂಲಕ — ಅಂತಿಮ ಉತ್ತರ ನೀಡುವ ಮೊದಲು.

ಉತ್ಪಾದನೆಯಲ್ಲಿ, ನೀವು in-memory `TRAVEL_KNOWLEDGE_BASE` ಅನ್ನು ರಿಯಲ್ ಆಜರ್ AI ಸರ್ಚ್ ಸೂಚ್ಯಂಕದಿಂದ ಬದಲಾಯಿಸುತ್ತೀರಿ, ಇದು ದೊಡ್ಡ ಮಟ್ಟದ ಪ್ರಯಾಣದ ದಾಖಲೆಗಳ ಪುರಸ್ಕಾರವನ್ನು ನಿರ್ವಹಿಸುತ್ತದೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ನಿರಾಕರಣೆ**:
ಈ ದಸ್ತಾವೇಜನ್ನು AI ಭಾಷಾಂತರ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಯಾಗಲು ಪ್ರಯತ್ನಿಸುವುದಾದರೂ, ಸ್ವಯಂಚಾಲಿತ ಭಾಷಾಂತರಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸತ್ಯತೆಯುಂಟಾಗಬಹುದು ಎಂದು ದಯವಿಟ್ಟು ಗಮನಿಸಿ. ಮೂಲ ದಸ್ತಾವೇಜನ್ನು ಅದರ ಸ್ವದೇಶಿ ಭಾಷೆಯಲ್ಲಿ ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. અગತ್ಯ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಭಾಷಾಂತರವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗಿದೆ. ಈ ಭಾಷಾಂತರದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗಬಹುದಾದ ಯಾವುದೇ ತಪ್ಪು ಅರಿವೋ ಅಥವಾ ತಪ್ಪು ವಿವರಣೆಗೂ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
